In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os
os.listdir("/content/drive/MyDrive")
os.listdir("/content/drive/MyDrive/NETRA")

In [ ]:
BASE_DIR = "/content/drive/MyDrive/NETRA"

VIDEO_DIR = BASE_DIR
DATASET_DIR = BASE_DIR + "/Traffic Video Analysis.v8i.yolov8"

In [ ]:
pip install ultralytics

In [ ]:
%%writefile traffic.yaml
path: /content/drive/MyDrive/NETRA/Traffic Video Analysis.v8i.yolov8

train: train/images
val: valid/images
test: test/images

names:
  0: car
  1: bike
  2: bus
  3: truck
  4: ambulance
  5: fire_truck
  6: police

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="traffic.yaml",
    epochs=50,
    imgsz=640,
    batch=16
)

In [ ]:
VIDEO_DIR = "/content/drive/MyDrive/NETRA"

import os
print(os.listdir(VIDEO_DIR))

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # lightweight & fast

In [ ]:
import cv2

video_path = VIDEO_DIR + "/cctv052x2004080.mp4"  # change exact filename
cap = cv2.VideoCapture(video_path)

vehicle_classes = [2, 3, 5, 7]  # car, motorcycle, bus, truck
frame_count = 0
vehicle_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    if frame_count % 10 != 0:  # skip frames for speed
        continue

    results = model(frame, conf=0.4)

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])
            if cls in vehicle_classes:
                vehicle_count += 1

cap.release()

print("Total detected vehicles:", vehicle_count)

In [ ]:
import cv2
from ultralytics import YOLO

video_path = "/content/drive/MyDrive/NETRA/your_video.mp4"  # change this
model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(video_path)

vehicle_classes = [2, 3, 5, 7]  # car, bike, bus, truck
vehicle_count = 0
frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Convert BGR → RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # YOLO inference
    results = model(frame_rgb, conf=0.25)

    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls in vehicle_classes:
                    vehicle_count += 1

cap.release()

print("Total frames read:", frame_count)
print("Total detected vehicles:", vehicle_count)

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/NETRA"))

In [ ]:
video_path = "/content/drive/MyDrive/NETRA/IndianRoad_demo.mp4"

In [ ]:
import cv2

cap = cv2.VideoCapture(video_path)
print("Opened:", cap.isOpened())

ret, frame = cap.read()
print("First frame read:", ret)
print("Frame shape:", frame.shape if ret else None)

cap.release()

In [ ]:
import cv2
from ultralytics import YOLO

video_path = "/content/drive/MyDrive/NETRA/IndianRoad_demo.mp4"
model = YOLO("yolov8n.pt")

cap = cv2.VideoCapture(video_path)

vehicle_classes = [2, 3, 5, 7]  # car, bike, bus, truck
total_vehicles = 0
processed_frames = 0

FRAME_SKIP = 5     # process every 5th frame
MAX_FRAMES = 300   # safety limit

frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret or processed_frames >= MAX_FRAMES:
        break

    frame_id += 1

    # Skip frames for speed
    if frame_id % FRAME_SKIP != 0:
        continue

    processed_frames += 1

    # Resize for faster inference
    frame = cv2.resize(frame, (640, 640))

    results = model.predict(
        source=frame,
        conf=0.3,
        imgsz=640,
        device="cpu",
        stream=False,
        verbose=False
    )

    frame_vehicle_count = 0
    for r in results:
        if r.boxes is not None:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id in vehicle_classes:
                    frame_vehicle_count += 1

    total_vehicles += frame_vehicle_count

cap.release()

density = total_vehicles / processed_frames if processed_frames > 0 else 0

print("Processed frames:", processed_frames)
print("Total detected vehicles:", total_vehicles)
print("Average vehicle density (vehicles/frame):", round(density, 3))

In [ ]:
import csv

with open("traffic_density.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Video", "Frames", "Total Vehicles", "Avg Density"])
    writer.writerow(["IndianRoad_demo.mp4", frame_count, total_vehicles, round(density, 3)])

print("CSV saved: traffic_density.csv")

Extract frames from video

In [ ]:
import cv2
import os

video_path = "/content/drive/MyDrive/NETRA/IndianRoad_demo.mp4"
out_dir = "frames/train/images"
os.makedirs(out_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

frame_id = 0
saved = 0
SKIP = 10   # extract every 10th frame

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1
    if frame_id % SKIP != 0:
        continue

    filename = f"frame_{saved:05d}.jpg"
    cv2.imwrite(os.path.join(out_dir, filename), frame)
    saved += 1

cap.release()
print("Frames extracted:", saved)

In [ ]:
import os

BASE = "/content/drive/MyDrive/NETRA/frames"

folders = [
    "train/images", "train/labels",
    "valid/images", "valid/labels"
]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("YOLO folder structure created ")

In [ ]:
import cv2
import os

video_path = "/content/drive/MyDrive/NETRA/IndianRoad_demo.mp4"
out_dir = "/content/drive/MyDrive/NETRA/frames/train/images"

cap = cv2.VideoCapture(video_path)

frame_id = 0
skip = 5   # take 1 frame every 5 frames

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % skip == 0:
        filename = f"road_{frame_id}.jpg"
        cv2.imwrite(os.path.join(out_dir, filename), frame)

    frame_id += 1

cap.release()
print("Frames extracted ")

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/NETRA"))

In [ ]:
video.endswith((".mp4", ".avi"))

In [ ]:
for v in os.listdir(VIDEO_DIR):
    print(v, "->", v.lower().endswith((".mp4", ".avi")))

In [ ]:
import os
import cv2

VIDEO_DIR = "/content/drive/MyDrive/NETRA"
OUTPUT_DIR = "/content/drive/MyDrive/NETRA/frames_all"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import os

BASE = "/content/drive/MyDrive/NETRA/yolo_dataset"

for split in ["train", "val"]:
    os.makedirs(f"{BASE}/images/{split}", exist_ok=True)
    os.makedirs(f"{BASE}/labels/{split}", exist_ok=True)

In [ ]:
%%writefile /content/drive/MyDrive/NETRA/traffic.yaml
path: /content/drive/MyDrive/NETRA/yolo_dataset

train: images/train
val: images/val

names:
  0: car
  1: bike
  2: bus
  3: truck
  4: ambulance
  5: fire_truck
  6: police

In [ ]:
!cat /content/drive/MyDrive/NETRA/traffic.yaml

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/drive/MyDrive/NETRA/traffic.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device="cpu"
)

In [ ]:
import os

base = "/content/drive/MyDrive/NETRA/yolo_dataset"
for p in [
    "train/images", "train/labels",
    "valid/images", "valid/labels"
]:
    print(p, "→", os.path.exists(os.path.join(base, p)))

In [ ]:
import os

base = "/content/drive/MyDrive/NETRA/yolo_dataset"

folders = [
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels"
]

for f in folders:
    path = os.path.join(base, f)
    os.makedirs(path, exist_ok=True)
    print("Created:", path)

In [ ]:
for p in folders:
    print(p, "→", os.path.exists(os.path.join(base, p)))

In [ ]:
import os

print("Train images:", len(os.listdir("/content/drive/MyDrive/NETRA/yolo_dataset/train/images")))
print("Train labels:", len(os.listdir("/content/drive/MyDrive/NETRA/yolo_dataset/train/labels")))
print("Valid images:", len(os.listdir("/content/drive/MyDrive/NETRA/yolo_dataset/valid/images")))
print("Valid labels:", len(os.listdir("/content/drive/MyDrive/NETRA/yolo_dataset/valid/labels")))

In [ ]:
import cv2
import os

VIDEO_DIR = "/content/drive/MyDrive/NETRA/video"
OUT_DIR = "/content/drive/MyDrive/NETRA/frames_all"

os.makedirs(OUT_DIR, exist_ok=True)

FRAME_GAP = 10  # take 1 frame every 10 frames

for video in os.listdir(VIDEO_DIR):
    if not video.endswith((".mp4", ".avi", ".mov")):
        continue

    cap = cv2.VideoCapture(os.path.join(VIDEO_DIR, video))
    count = 0
    saved = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % FRAME_GAP == 0:
            name = f"{video}_{saved}.jpg"
            cv2.imwrite(os.path.join(OUT_DIR, name), frame)
            saved += 1

        count += 1

    cap.release()
    print(f"{video}: saved {saved} frames")

In [ ]:
FRAME_GAP = 3

In [ ]:
import os
print("Total extracted frames:", len(os.listdir("/content/drive/MyDrive/NETRA/frames_all")))

In [ ]:
import os

base = "/content/drive/MyDrive/NETRA"
for root, dirs, files in os.walk(base):
    if any(f.endswith(".txt") for f in files):
        print("Found labels in:", root)
        break

In [ ]:
for p in [
    "train/images", "train/labels",
    "valid/images", "valid/labels"
]:
    full = f"/content/drive/MyDrive/NETRA/yolo_dataset/{p}"
    print(p, "→", len(os.listdir(full)))

to find extra label files

In [ ]:
import os

img_dir = "/content/drive/MyDrive/NETRA/yolo_dataset/train/images"
lbl_dir = "/content/drive/MyDrive/NETRA/yolo_dataset/train/labels"

images = set(os.path.splitext(f)[0] for f in os.listdir(img_dir))
labels = set(os.path.splitext(f)[0] for f in os.listdir(lbl_dir))

extra_labels = labels - images
extra_images = images - labels

print("Extra label files:", extra_labels)
print("Images without labels:", extra_images)

Delete EXTRA LABEL FILES

In [ ]:
import os

lbl_dir = "/content/drive/MyDrive/NETRA/yolo_dataset/train/labels"

extra_labels = {
 'video_17_frame_1052_1_jpg.rf.12c7446e50d567d58b6f2e8bc27196c5 (1)',
 'video_17_frame_2072_jpg.rf.2d26dfe054dbb2e8ceb5cb234a602c35 (1)',
 'video_20_frame_1208_jpg.rf.06b029ea434a94311b84e7b47ddf03fe (1)',
 'video_8_frame_1071_jpg.rf.b979316228ad9d282a7db42a85184928 (1)',
 'video_14_frame_1748_jpg.rf.2d154ad3ed42de587a6319def28b0fca (1)',
 'video_2_frame_240_1_jpg.rf.d74921e1787ff945b87f12816ceb8c80 (1)'
}

for name in extra_labels:
    path = f"{lbl_dir}/{name}.txt"
    if os.path.exists(path):
        os.remove(path)
        print("Deleted label:", path)

Delete IMAGES WITHOUT LABELS

In [ ]:
img_dir = "/content/drive/MyDrive/NETRA/yolo_dataset/train/images"

extra_images = {
 'video_15_frame_1893_jpg.rf.ef0aaf86a4be23a7e2d510630a1b8cdf (1)',
 'video_23_frame_2798_2_jpg.rf.d27104861a98b7033bd0bc5045ccf720 (1)',
 'video_23_frame_1416_1_jpg.rf.a3b3596f97ade7c3f680dda6bb8eeeaf (1)',
 'video_4_frame_493_jpg.rf.6111a32680c5228c3759102adf39e94f (1)',
 'video_6_frame_361_1_jpg.rf.fa5d6f41f70175c44d7f0e0257e32eb7 (1)'
}

for name in extra_images:
    path = f"{img_dir}/{name}.jpg"
    if os.path.exists(path):
        os.remove(path)
        print("Deleted image:", path)

In [ ]:
print("Train images:", len(os.listdir(img_dir)))
print("Train labels:", len(os.listdir(lbl_dir)))

To make sure labels are valid

In [ ]:
%%writefile /content/drive/MyDrive/NETRA/yolo_dataset/data.yaml
path: /content/drive/MyDrive/NETRA/yolo_dataset

train: train/images
val: valid/images

nc: 6

names:
  0: car
  1: bus
  2: truck
  3: bike
  4: auto
  5: person

In [ ]:
!yolo detect train \
  model=yolov8n.pt \
  data=/content/drive/MyDrive/NETRA/yolo_dataset/data.yaml \
  epochs=20 \
  project=/content/drive/MyDrive/NETRA/runs \
  name=netra_v1

In [ ]:
ls /content/drive/MyDrive/NETRA/runs/

In [ ]:
ls /content/drive/MyDrive/NETRA/runs/netra_v13/weights

Run YOLO on ALL traffic videos (Batch Inference)

In [ ]:
!yolo detect predict \
 model=/content/drive/MyDrive/NETRA/runs/netra_v13/weights/best.pt \
 source=/content/drive/MyDrive/NETRA/North.mp4 \
 save=True

count them per frame / lane

Traffic density classification

In [ ]:
vehicle_count = 0

for r in results:
    for c in r.boxes.cls:
        cls_name = model.names[int(c)]
        if cls_name in ["car", "bus", "truck", "bike", "auto"]:
            vehicle_count += 1


In [ ]:
from ultralytics import YOLO
import cv2

model = YOLO("/content/drive/MyDrive/NETRA/runs/netra_v13/weights/best.pt")
cap = cv2.VideoCapture("/content/drive/MyDrive/NETRA/North.mp4")

vehicle_classes = ["car", "bus", "truck", "bike", "auto"]

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)[0]

    count = 0
    for box in results.boxes:
        cls = int(box.cls[0])
        label = results.names[cls]
        if label in vehicle_classes:
            count += 1

    print("Vehicles in frame:", count)

cap.release()

In [ ]:
if vehicle_count <= 5:
    traffic_level = "LOW"
elif vehicle_count <= 15:
    traffic_level = "MEDIUM"
else:
    traffic_level = "HIGH"

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/NETRA/runs/netra_v13/weights/best.pt")
import cv2

video_path = "/content/drive/MyDrive/NETRA/North.mp4"
cap = cv2.VideoCapture(video_path)

assert cap.isOpened(), "Video not opened"
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame, conf=0.4, verbose=False)

In [ ]:
    vehicle_count = 0

    for r in results:
        if r.boxes is not None:
            for c in r.boxes.cls:
                cls_name = model.names[int(c)]
                if cls_name in ["car", "bus", "truck", "bike", "auto"]:
                    vehicle_count += 1

In [ ]:
    if vehicle_count <= 5:
        traffic_level = "LOW"
    elif vehicle_count <= 15:
        traffic_level = "MEDIUM"
    else:
        traffic_level = "HIGH"

In [ ]:
    print(f"Vehicles: {vehicle_count} | Traffic: {traffic_level}")

In [ ]:
cap.release()
print(" Video processing completed")

In [ ]:
import cv2

cap = cv2.VideoCapture("/content/drive/MyDrive/NETRA/North.mp4")
frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Frames:", frames)
print("FPS:", fps)

cap.release()

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/NETRA/runs/netra_v13/weights/best.pt")

cap = cv2.VideoCapture("/content/drive/MyDrive/NETRA/North.mp4")

total_vehicles = 0
processed_frames = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, conf=0.4, verbose=False)

    vehicle_count = 0
    for r in results:
        for cls in r.boxes.cls:
            vehicle_count += 1

    total_vehicles += vehicle_count
    processed_frames += 1

cap.release()

avg_vehicles = total_vehicles / processed_frames

print("Average vehicles per frame:", round(avg_vehicles, 2))

In [ ]:
if avg_vehicles <= 5:
    traffic_level = "LOW"
    green_time = 20
elif avg_vehicles <= 15:
    traffic_level = "MEDIUM"
    green_time = 40
else:
    traffic_level = "HIGH"
    green_time = 60

print("Traffic Level:", traffic_level)
print("Green Signal Time:", green_time, "seconds")

In [ ]:
print("Average vehicles per frame:", avg_vehicles)

In [ ]:
if avg_vehicles <= 2:
    traffic_level = "LOW"
    green_time = 20
elif avg_vehicles <= 5:
    traffic_level = "MEDIUM"
    green_time = 40
else:
    traffic_level = "HIGH"
    green_time = 60

In [ ]:
import time

print("GREEN signal ON")
time.sleep(green_time)

print("YELLOW signal ON")
time.sleep(3)

print("RED signal ON")

In [ ]:
def calculate_traffic():

    vehicle_counts = []

    for frame in video_frames:   # or your frame processing loop

        results = model(frame)

        vehicle_count = 0

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ["car", "bus", "truck", "motorcycle"]:
                    vehicle_count += 1

        vehicle_counts.append(vehicle_count)

    avg_vehicles = sum(vehicle_counts) / len(vehicle_counts)

    print("Average vehicles per frame:", avg_vehicles)

    return avg_vehicles

In [ ]:
def calculate_traffic(cap, frame_limit=300):  # ~10 sec at 30 FPS

    vehicle_counts = []
    count = 0

    while count < frame_limit:
        ret, frame = cap.read()
        if not ret:
            break

        # detection logic here

        vehicle_counts.append(vehicle_count)
        count += 1

    if len(vehicle_counts) == 0:
        return 0

    return sum(vehicle_counts) / len(vehicle_counts)

In [ ]:
cap = cv2.VideoCapture("North.mp4")

while cap.isOpened():

    avg_vehicles = calculate_traffic(cap)

    if avg_vehicles <= 2:
        green_time = 20
    elif avg_vehicles <= 5:
        green_time = 40
    else:
        green_time = 60

    print("GREEN signal ON")
    time.sleep(green_time)

    print("YELLOW signal ON")
    time.sleep(3)

    print("RED signal ON")

cap.release()

In [ ]:
restart_count = 0
max_restarts = 1

In [ ]:
import cv2
import time
from google.colab.patches import cv2_imshow

In [ ]:
cap = cv2.VideoCapture("North.mp4")

restart_count = 0
max_restarts = 1

while True:

    avg_vehicles = calculate_traffic(cap)

    if avg_vehicles == -1:
        if restart_count < max_restarts:
            print("Restarting Video...")
            cap.release()
            cap = cv2.VideoCapture("North.mp4")
            restart_count += 1
            continue
        else:
            print("Video Fully Completed")
            break

    if avg_vehicles <= 2:
        green_time = 20
    elif avg_vehicles <= 5:
        green_time = 40
    else:
        green_time = 60

    print("Average vehicles:", avg_vehicles)

    print("GREEN signal ON")
    time.sleep(green_time)

    print("YELLOW signal ON")
    time.sleep(3)

    print("RED signal ON")

cap.release()

Updated calculate_traffic

In [ ]:
def calculate_traffic(cap, frame_limit=100):

    vehicle_counts = []
    count = 0

    while count < frame_limit:
        ret, frame = cap.read()

        if not ret:
            return -1

        vehicle_count = 0

        results = model(frame)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ["car", "bus", "truck", "motorcycle"]:
                    vehicle_count += 1

                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    cv2.putText(frame, class_name,
                                (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.6,
                                (0, 255, 0),
                                2)

        vehicle_counts.append(vehicle_count)
        count += 1

        cv2.putText(frame,
                    f"Vehicles: {vehicle_count}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3)

        cv2_imshow(frame)

        time.sleep(0.03)

    if len(vehicle_counts) == 0:
        return 0

    return sum(vehicle_counts) / len(vehicle_counts)

Main Execution Loop

In [ ]:
cap = cv2.VideoCapture("North.mp4")

while cap.isOpened():

    avg_vehicles = calculate_traffic(cap)

    if avg_vehicles == -1:
        print("Video Completed")
        break

    if avg_vehicles <= 2:
        green_time = 20
    elif avg_vehicles <= 5:
        green_time = 40
    else:
        green_time = 60

    print("Average vehicles:", avg_vehicles)

    print("GREEN signal ON")
    time.sleep(2)

    print("YELLOW signal ON")
    time.sleep(1)

    print("RED signal ON")

cap.release()

In [ ]:
cap = cv2.VideoCapture("North.mp4")

print("Video opened:", cap.isOpened())

In [ ]:
def calculate_traffic(cap, frame_limit=100):

    vehicle_counts = []
    count = 0

    while count < frame_limit:
        ret, frame = cap.read()

        if not ret:
            return -1

        vehicle_count = 0

        results = model(frame)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ["car", "bus", "truck", "motorcycle"]:
                    vehicle_count += 1

                    x1, y1, x2, y2 = map(int, box.xyxy[0])

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

                    cv2.putText(frame, class_name,
                                (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX,
                                0.6,
                                (0, 255, 0),
                                2)

        vehicle_counts.append(vehicle_count)
        count += 1

        cv2.putText(frame,
                    f"Vehicles: {vehicle_count}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0, 0, 255),
                    3)

        cv2_imshow(frame)

        time.sleep(0.03)

    if len(vehicle_counts) == 0:
        return 0

    return sum(vehicle_counts) / len(vehicle_counts)

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/NETRA/runs/netra_v13/weights/best.pt")

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

video_path = "/content/drive/MyDrive/NETRA/North.mp4"

cap = cv2.VideoCapture(video_path)

print("Video opened:", cap.isOpened())

ret, frame = cap.read()
print("Frame read:", ret)

if ret:
    cv2_imshow(frame)
else:
    print("Could not read frame")

cap.release()

In [ ]:
import cv2
import time
from google.colab.patches import cv2_imshow

def calculate_traffic(cap, frame_limit=50):

    vehicle_counts = []
    count = 0

    while count < frame_limit:
        ret, frame = cap.read()

        if not ret:
            return -1

        vehicle_count = 0

        results = model(frame)

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ["car", "bus", "truck", "motorcycle"]:
                    vehicle_count += 1

                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

        vehicle_counts.append(vehicle_count)
        count += 1

        cv2.putText(frame,
                    f"Vehicles: {vehicle_count}",
                    (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,0,255),
                    3)

        cv2_imshow(frame)
        time.sleep(0.05)

    if len(vehicle_counts) == 0:
        return 0

    return sum(vehicle_counts) / len(vehicle_counts)


video_path = "/content/drive/MyDrive/NETRA/North.mp4"
cap = cv2.VideoCapture(video_path)

while cap.isOpened():

    avg_vehicles = calculate_traffic(cap)

    if avg_vehicles == -1:
        print("Video Completed")
        break

    if avg_vehicles <= 2:
        green_time = 20
    elif avg_vehicles <= 5:
        green_time = 40
    else:
        green_time = 60

    print("Average vehicles:", avg_vehicles)
    print("GREEN signal ON")
    time.sleep(2)

    print("YELLOW signal ON")
    time.sleep(1)

    print("RED signal ON")

cap.release()

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

In [ ]:
print(model.names)

Use Lower Confidence Threshold

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

video_path = "/content/drive/MyDrive/NETRA/North.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if ret:
    results = model(frame, conf=0.25)

    vehicle_count = 0

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])

            if cls in [2, 3, 5, 7]:   # car, motorcycle, bus, truck
                vehicle_count += 1

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    print("Vehicles detected:", vehicle_count)
    cv2_imshow(frame)

else:
    print("Could not read frame")

cap.release()

In [ ]:
results = model(frame, conf=0.25)

for r in results:
    for box in r.boxes:
        cls = int(box.cls[0])

        if cls in [2, 3, 5, 7]:   # car, motorcycle, bus, truck
            vehicle_count += 1

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

            label = model.names[cls]
            cv2.putText(frame, label,
                        (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0,255,0),
                        2)

TESTING FOR SOUTH DIRECTION

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

video_path = "/content/drive/MyDrive/NETRA/South.mp4"

cap = cv2.VideoCapture(video_path)

ret, frame = cap.read()

if ret:
    results = model(frame, conf=0.25)

    vehicle_count = 0

    for r in results:
        for box in r.boxes:
            cls = int(box.cls[0])

            if cls in [2, 3, 5, 7]:  # car, motorcycle, bus, truck
                vehicle_count += 1

                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    print("Vehicles detected:", vehicle_count)
    cv2_imshow(frame)

else:
    print("Could not read frame")

cap.release()

In [ ]:
def test_video(video_path):

    cap = cv2.VideoCapture(video_path)
    total_count = 0
    frame_count = 0

    while frame_count < 50:   # test first 50 frames
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, conf=0.25)
        vehicle_count = 0

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                if cls in [2, 3, 5, 7]:
                    vehicle_count += 1

        total_count += vehicle_count
        frame_count += 1

    cap.release()

    if frame_count == 0:
        print("No frames read")
        return

    print("Video:", video_path)
    print("Average vehicles per frame:", total_count/frame_count)

In [ ]:
test_video("/content/drive/MyDrive/NETRA/South.mp4")
test_video("/content/drive/MyDrive/NETRA/East.mp4")
test_video("/content/drive/MyDrive/NETRA/Weast.mp4")

Average for All 4 Directions

In [ ]:
videos = {
    "North": "/content/drive/MyDrive/NETRA/North.mp4",
    "South": "/content/drive/MyDrive/NETRA/South.mp4",
    "East": "/content/drive/MyDrive/NETRA/East.mp4",
    "West": "/content/drive/MyDrive/NETRA/Weast.mp4"
}

traffic_data = {}

for direction, path in videos.items():
    avg = test_video(path)
    traffic_data[direction] = avg

In [ ]:
highest_direction = max(traffic_data, key=traffic_data.get)

print("Traffic Data:", traffic_data)
print("Highest Traffic Direction:", highest_direction)

In [ ]:
print(traffic_data)

In [ ]:
traffic_data = {}

videos = {
    "North": "/content/drive/MyDrive/NETRA/North.mp4",
    "South": "/content/drive/MyDrive/NETRA/South.mp4",
    "East": "/content/drive/MyDrive/NETRA/East.mp4",
    "West": "/content/drive/MyDrive/NETRA/Weast.mp4"
}

for direction, path in videos.items():
    avg = test_video(path)
    traffic_data[direction] = avg

print("Traffic Data:", traffic_data)

test video

In [ ]:
import cv2

def test_video(video_path, frame_limit=100):

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video:", video_path)
        return 0

    vehicle_counts = []
    frame_count = 0

    while frame_count < frame_limit:

        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame)

        vehicle_count = 0

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                class_name = model.names[cls]

                if class_name in ["car", "bus", "truck", "motorcycle"]:
                    vehicle_count += 1

        vehicle_counts.append(vehicle_count)
        frame_count += 1

    cap.release()

    if len(vehicle_counts) == 0:
        return 0

    avg = sum(vehicle_counts) / len(vehicle_counts)

    print("Video:", video_path)
    print("Average vehicles per frame:", round(avg, 2))

    return avg

In [ ]:
highest_direction = max(traffic_data, key=traffic_data.get)

print("\nHighest Traffic Direction:", highest_direction)

Signal Control

In [ ]:
for direction in traffic_data:

    if direction == highest_direction:
        print(direction, "→ GREEN for 60 seconds")
    else:
        print(direction, "→ RED")

In [ ]:
from google.colab.patches import cv2_imshow
import cv2

cap = cv2.VideoCapture("/content/drive/MyDrive/NETRA/North.mp4")

ret, frame = cap.read()

results = model(frame)

for r in results:
    frame = r.plot()

cv2_imshow(frame)

cap.release()

In [ ]:
traffic_data = {}

videos = {
    "North": "/content/drive/MyDrive/NETRA/North.mp4",
    "South": "/content/drive/MyDrive/NETRA/South.mp4",
    "East": "/content/drive/MyDrive/NETRA/East.mp4",
    "West": "/content/drive/MyDrive/NETRA/Weast.mp4"
}

for direction, path in videos.items():

    avg = test_video(path)   # your vehicle detection function

    traffic_data[direction] = avg

print("\nTraffic Data:", traffic_data)

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

def test_video(video_path, direction):

    cap = cv2.VideoCapture(video_path)

    frame_count = 0
    total_vehicles = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        results = model(frame)

        vehicle_count = 0

        for r in results:
            for box in r.boxes:

                cls = int(box.cls[0])
                name = model.names[cls]

                if name in ["car","bus","truck","motorcycle"]:
                    vehicle_count += 1

            frame = r.plot()

        total_vehicles += vehicle_count
        frame_count += 1

        # Display vehicle count
        cv2.putText(frame,
                    f"Vehicles: {vehicle_count}",
                    (20,50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (0,255,0),
                    2)

        # Display direction
        cv2.putText(frame,
                    f"Direction: {direction}",
                    (20,100),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1,
                    (255,0,0),
                    2)

        cv2_imshow(frame)

        if frame_count > 50:
            break

    cap.release()

    avg = total_vehicles / frame_count

    print("Direction:", direction)
    print("Average vehicles:", round(avg,2))

    return avg

In [ ]:
traffic_data = {}

videos = {
    "North": "/content/drive/MyDrive/NETRA/North.mp4",
    "South": "/content/drive/MyDrive/NETRA/South.mp4",
    "East": "/content/drive/MyDrive/NETRA/East.mp4",
    "West": "/content/drive/MyDrive/NETRA/Weast.mp4"
}

for direction, path in videos.items():

    avg = test_video(path, direction)

    traffic_data[direction] = avg

In [ ]:
pip install osmnx networkx

In [ ]:
import osmnx as ox
import networkx as nx

place = "Dehradun, India"

G = ox.graph_from_place(place, network_type='drive')

ox.plot_graph(G)

In [ ]:
ox.plot_graph(G)

In [ ]:
import sqlite3

conn = sqlite3.connect("/content/drive/MyDrive/NETRA/traffic.db")
cursor = conn.cursor()

In [ ]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS nodes (
    node_id INTEGER PRIMARY KEY,
    lat REAL,
    lon REAL,
    traffic_density REAL,
    emergency INTEGER
)
""")

conn.commit()
print(" Table created successfully")

In [ ]:
cursor.execute("SELECT * FROM nodes")
print(cursor.fetchall())

In [ ]:
import osmnx as ox
import networkx as nx

place = "Dehradun, India"
G = ox.graph_from_place(place, network_type='drive')

print("Nodes:", len(G.nodes))
print("Edges:", len(G.edges))

In [ ]:
 import sqlite3

conn = sqlite3.connect("/content/drive/MyDrive/NETRA/traffic.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS nodes (
    node_id INTEGER PRIMARY KEY,
    lat REAL,
    lon REAL,
    traffic_density REAL DEFAULT 0,
    emergency INTEGER DEFAULT 0
)
""")

# Insert nodes
for node, data in G.nodes(data=True):
    cursor.execute("""
    INSERT OR IGNORE INTO nodes (node_id, lat, lon)
    VALUES (?, ?, ?)
    """, (node, data['y'], data['x']))

conn.commit()
print(" Nodes stored in DB")

In [ ]:
lat = 30.3165
lon = 78.0322

In [ ]:
nearest_node = ox.distance.nearest_nodes(G, lon, lat)

In [ ]:
lat = 30.3165
lon = 78.0322

density = 3.5

nearest_node = ox.distance.nearest_nodes(G, lon, lat)

cursor.execute("""
UPDATE nodes
SET traffic_density = ?
WHERE node_id = ?
""", (density, nearest_node))

conn.commit()

print(" Traffic mapped to node:", nearest_node)

In [ ]:
!pip install inference-sdk

In [ ]:
from inference_sdk import InferenceHTTPClient

emergency part

In [ ]:
!pip install librosa
!pip install tensorflow

In [ ]:
base_path = "/content/drive/MyDrive/archive/Emergency_Vehicles"

In [ ]:
%%writefile /content/emergency.yaml

path: /content/drive/MyDrive/archive/Emergency_Vehicles

train: train/images
val: test/images

names:
  0: ambulance
  1: fire_truck
  2: police

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/NETRA/archive"))
print(os.listdir("/content/drive/MyDrive/NETRA/archive/Emergency_Vehicles"))
BASE_PATH = "/content/drive/MyDrive/NETRA/archive/Emergency_Vehicles"

In [ ]:
import pandas as pd
import os

BASE_PATH = "/content/drive/MyDrive/NETRA/archive/Emergency_Vehicles"

df = pd.read_csv(BASE_PATH + "/train.csv")

print(df.head())

In [ ]:
print(df.columns)

In [ ]:
df['image_names'] = df['image_names'].str.replace(" ", "")

In [ ]:
import cv2
import numpy as np

IMG_SIZE = 128

def load_data(df):

    X = []
    y = []

    for i in range(len(df)):
        img_name = df.iloc[i]['image_names']
        label = df.iloc[i]['emergency_or_not']

        img_path = os.path.join(BASE_PATH, "train", img_name)

        if not os.path.exists(img_path):
            continue

        img = cv2.imread(img_path)

        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        X.append(img)
        y.append(label)

    return np.array(X)/255.0, np.array(y)

X, y = load_data(df)

print("Data loaded:", X.shape)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model_img = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(1, activation='sigmoid')
])

model_img.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model_img.fit(X, y, epochs=5, batch_size=32)

In [ ]:
def detect_emergency_image(frame):

    img = cv2.resize(frame, (128,128))
    img = img / 255.0
    img = np.reshape(img, (1,128,128,3))

    pred = model_img.predict(img)[0][0]

    if pred > 0.5:
        return True
    return False

In [ ]:
import librosa
import numpy as np

def extract_features(file):
    y, sr = librosa.load(file, duration=3)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)

In [ ]:
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_audio = []
y_audio = []

base_audio = BASE_PATH + "/sounds"

labels_map = {
    "ambulance": 0,
    "firetruck": 1,
    "traffic": 2
}

for folder in labels_map:
    folder_path = os.path.join(base_audio, folder)

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)

        try:
            features = extract_features(file_path)
            X_audio.append(features)
            y_audio.append(labels_map[folder])
        except:
            pass

X_audio = np.array(X_audio)
y_audio = np.array(y_audio)

X_train, X_test, y_train, y_test = train_test_split(X_audio, y_audio, test_size=0.2)

audio_model = RandomForestClassifier()
audio_model.fit(X_train, y_train)

print("Audio Accuracy:", audio_model.score(X_test, y_test))

In [ ]:
def detect_siren(audio_file):

    features = extract_features(audio_file).reshape(1, -1)
    pred = audio_model.predict(features)[0]

    if pred in [0,1]:
        return True
    return False

In [ ]:
def detect_emergency(frame, audio_file):

    image_flag = detect_emergency_image(frame)
    audio_flag = detect_siren(audio_file)

    # BEST PRACTICAL LOGIC
    if audio_flag:
        return True

    if image_flag:
        return True

    return False

In [ ]:
!apt-get install ffmpeg -y

In [ ]:
import os

video_path = "/content/drive/MyDrive/NETRA/North.mp4"
audio_path = "/content/audio.wav"

os.system(f"ffmpeg -i '{video_path}' -q:a 0 -map a '{audio_path}' -y")

print("Audio extracted")

In [ ]:
emergency = detect_emergency(frame, audio_path)
audio_flag = detect_siren(audio_path)

In [ ]:
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image_flag = detect_emergency_image(frame)

    # Combine
    if audio_flag or image_flag:
        print(" EMERGENCY DETECTED")
        break

cap.release()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2)

model_img.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

In [ ]:
model_img.save("/content/drive/MyDrive/NETRA/emergency_cnn.h5")
import joblib
joblib.dump(audio_model, "/content/drive/MyDrive/NETRA/audio_model.pkl")

In [ ]:
videos = {
    "North": "/content/drive/MyDrive/NETRA/North.mp4",
    "South": "/content/drive/MyDrive/NETRA/South.mp4",
    "East": "/content/drive/MyDrive/NETRA/east.mp4",
    "West": "/content/drive/MyDrive/NETRA/Weast.mp4"
}

traffic_data = {}

In [ ]:
for direction, path in videos.items():

    cap = cv2.VideoCapture(path)

    total_vehicles = 0
    frames = 0

    emergency_flag = False

    while cap.isOpened() and frames < 50:  # limit frames
        ret, frame = cap.read()
        if not ret:
            break

        frames += 1

        # VEHICLE DETECTION (your YOLO)
        results = model(frame)
        count = 0

        for r in results:
            for box in r.boxes:
                cls = int(box.cls[0])
                label = model.names[cls]

                if label in ["car", "bus", "truck", "bike"]:
                    count += 1

        total_vehicles += count

        #  EMERGENCY DETECTION
        if detect_emergency_image(frame):
            emergency_flag = True

    cap.release()

    avg_vehicles = total_vehicles / frames if frames > 0 else 0

    traffic_data[direction] = {
        "vehicles": avg_vehicles,
        "emergency": emergency_flag
    }

In [ ]:
def detect_emergency_image(frame):

    img = cv2.resize(frame, (128,128))
    img = img / 255.0
    img = np.reshape(img, (1,128,128,3))

    pred = model_img.predict(img, verbose=0)[0][0]
    return pred > 0.5

In [ ]:
frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    if frame_id % 5 != 0:
        continue   # skip frames

    image_flag = detect_emergency_image(frame)

In [ ]:
traffic_data = {}

for direction, path in videos.items():

    print(f"\n Processing {direction} ...")

    cap = cv2.VideoCapture(path)

    if not cap.isOpened():
        print(" Video not opened:", path)
        continue

    total_vehicles = 0
    frames = 0
    emergency_flag = False

    while cap.isOpened() and frames < 50:

        ret, frame = cap.read()
        if not ret:
            print(" Frame read failed")
            break

        frames += 1

        # VEHICLE DETECTION
        results = model(frame)

        count = 0

        for r in results:
            if r.boxes is None:
                continue

            for box in r.boxes:
                cls = int(box.cls[0])
                label = model.names[cls]

                if label in ["car", "bus", "truck", "bike"]:
                    count += 1

        total_vehicles += count

        # EMERGENCY DETECTION
        if detect_emergency_image(frame):
            emergency_flag = True
            print(" Emergency detected in", direction)
            break

    cap.release()

    # avoid divide-by-zero
    avg_vehicles = total_vehicles / frames if frames > 0 else 0

    print(f" {direction} → Vehicles:", round(avg_vehicles, 2),
          "| Emergency:", emergency_flag)

    traffic_data[direction] = {
        "vehicles": avg_vehicles,
        "emergency": emergency_flag
    }

In [ ]:
def detect_emergency_image(frame):

    results = model(frame, conf=0.5)  # IMPORTANT threshold

    for r in results:
        for box in r.boxes:

            cls = int(box.cls[0])
            conf = float(box.conf[0])

            label = model.names[cls]

            # ONLY high confidence vehicles
            if conf > 0.6 and label in ["bus", "truck"]:

                return True

    return False

In [ ]:
emergency_count = 0
total_frames = 0

In [ ]:
for direction, path in videos.items():

    print(f"\nProcessing {direction} ...")

    cap = cv2.VideoCapture(path)

    total_vehicles = 0
    emergency_count = 0
    frames = 0

    while cap.isOpened() and frames < 50:

        ret, frame = cap.read()
        if not ret:
            break

        frames += 1

        # VEHICLE COUNT
        results = model(frame, conf=0.5)

        count = 0

        for r in results:
            for box in r.boxes:

                cls = int(box.cls[0])
                conf = float(box.conf[0])

                label = model.names[cls]

                if label in ["car", "bus", "truck", "bicycle"]:
                    count += 1

                # emergency check
                if conf > 0.6 and label in ["bus", "truck"]:
                    emergency_count += 1

        total_vehicles += count

    cap.release()

    avg_vehicles = total_vehicles / frames if frames > 0 else 0

    # FINAL DECISION LOGIC
    emergency_flag = emergency_count > 5   # IMPORTANT FIX

    traffic_data[direction] = {
        "vehicles": avg_vehicles,
        "emergency": emergency_flag
    }

    print(f"{direction} → Vehicles: {avg_vehicles:.2f} | Emergency: {emergency_flag}")

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="emergency.yaml",
    epochs=30,
    imgsz=640
)

In [ ]:
def is_emergency_vehicle(cls_name, conf):
    emergency_classes = ["ambulance", "fire_truck", "police"]

    return cls_name in emergency_classes and conf > 0.5

In [ ]:
for r in results:
    for box in r.boxes:

        cls = int(box.cls[0])
        conf = float(box.conf[0])
        label = model.names[cls]

        if label in ["car", "bus", "truck", "bicycle"]:
            count += 1

        if is_emergency_vehicle(label, conf):
            emergency_count += 1

In [ ]:
print("Detected:", label, conf)

In [ ]:
def is_emergency_vehicle(label, conf):
    return label in ["ambulance", "fire_truck", "police"]

In [ ]:
def is_emergency_vehicle(label, conf):
    return label in ["bus", "truck"] and conf > 0.6

def is_emergency_vehicle(label, conf, frame_density):
    return (
        (label in ["bus", "truck"] and conf > 0.6)
        or frame_density > 15  # congestion emergency condition
    )

In [ ]:
def apply_emergency_priority(graph, emergency_node):

    for node in graph:
        for neighbor in graph[node]:
            if node == emergency_node:
                graph[node][neighbor] *= 0.2   # prioritize route

In [ ]:
def get_live_traffic(video_results):

    traffic_data = {}

    for direction, result in video_results.items():

        total = result["vehicles"]

        # convert to density (normalize)
        density = min(total / 10, 1.0)

        traffic_data[direction] = {
            "vehicles": total,
            "density": density
        }

    return traffic_data

In [ ]:
video_results = {
    "North": {"vehicles": 12},
    "South": {"vehicles": 3},
    "East": {"vehicles": 1},
    "West": {"vehicles": 7}
}

In [ ]:
traffic_data = get_live_traffic(video_results)

In [ ]:
import sqlite3

conn = sqlite3.connect("traffic.db")
cursor = conn.cursor()

In [ ]:
import sqlite3

conn = sqlite3.connect("traffic.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS traffic_data (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    direction TEXT,
    vehicles REAL,
    emergency INTEGER,
    timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
""")

conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect("traffic.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

In [ ]:
conn = sqlite3.connect("traffic.db")
cursor = conn.cursor()

cursor.execute("""
INSERT INTO traffic_data (direction, vehicles, emergency)
VALUES (?, ?, ?)
""", ("North", 12, 1))

conn.commit()
conn.close()

In [ ]:
conn = sqlite3.connect("traffic.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM traffic_data")
print(cursor.fetchall())

conn.close()

In [ ]:
def get_live_traffic_from_db():
    conn = sqlite3.connect("traffic.db")
    cursor = conn.cursor()

    cursor.execute("""
        SELECT direction, vehicles, emergency
        FROM traffic_data
        ORDER BY id DESC
        LIMIT 4
    """)

    rows = cursor.fetchall()
    conn.close()

    traffic = {}

    for direction, vehicles, emergency in rows:
        traffic[direction] = {
            "vehicles": vehicles,
            "emergency": emergency
        }

    return traffic

In [ ]:
def update_weights(graph, traffic_data, alpha=2.0):

    new_graph = {}

    for node, neighbors in graph.items():
        new_graph[node] = {}

        for nbr, base_weight in neighbors.items():

            traffic = traffic_data.get(node, {}).get("vehicles", 0)

            emergency = traffic_data.get(node, {}).get("emergency", 0)

            new_weight = base_weight * (1 + alpha * traffic)

            if emergency:
                new_weight *= 0.3

            new_graph[node][nbr] = new_weight

    return new_graph

In [ ]:
traffic_data = get_live_traffic_from_db()
print("Traffic Data:", traffic_data)

In [ ]:
graph = {
    "A": {"B": 2, "C": 5},
    "B": {"A": 2, "D": 4},
    "C": {"A": 5, "D": 1},
    "D": {"B": 4, "C": 1}
}

In [ ]:
updated_graph = update_weights(graph, traffic_data)
print("Updated Graph:", updated_graph)

In [ ]:
import json

with open("/content/traffic.json", "w") as f:
    json.dump(traffic_data, f)

print("Saved traffic.json")

In [ ]:
from google.colab import files
files.download("/content/traffic.json")

In [ ]:
import json

with open("/content/traffic.json", "w") as f:
    json.dump(traffic_data, f)

from google.colab import files
files.download("/content/traffic.json")